# Setting up MISProjectHDF5 by ingesting files

In [ ]:
from misalign.model.project import MISProjectHDF5

In [ ]:
mis_project=MISProjectHDF5.build(
    mis_filepath="demo-project_a.mis.hdf5",
    ingest_image_filepaths=[f"../project_a/image_a{i+1:02d}.jpg" for i in range(10)],
    calibration_filepath="../project_a/scale_5x_1mm.miscal.json")

# Adding relations from existing project

In [ ]:
from misalign.model.project import MISProjectJSON

In [ ]:
mis_project_example=MISProjectJSON.load(
    mis_filepath="../project_a/project_a-relations-calibrated.mis.json")
example_relations=mis_project_example.get_relations()
[r.for_json() for r in example_relations][0]

In [ ]:
for relation in example_relations:
    mis_project.add_relation(relation)

In [ ]:
mis_project.save(project_hdf5path="MISContainer0/project_rel")

# Render from hdf5

In [ ]:
from misalign.model.project import MISProjectHDF5
import misalign.canvas.canvas_rectangular as cr

In [ ]:
mis_project=MISProjectHDF5.load(
    mis_filepath="project_a.mis.hdf5",
    project_hdf5path="MISContainer0/project_rel")
print(mis_project)

In [ ]:
origin_relative_offsets=cr.rectangular_solve_project(
    project=mis_project,
)
print("Origin-relative offsets")
display(origin_relative_offsets)
origin_relative_extents=cr.find_relative_extents_project(
    project=mis_project,
    origin_relative_offsets=origin_relative_offsets,)
canvas_extents, canvas_offsets=cr.resolve_extents(origin_relative_extents)
canvas_relative_offsets=cr.place_in_canvas(
    image_names=mis_project.get_image_names(),
    origin_relative_offsets=origin_relative_offsets,
    canvas_extents=canvas_extents,
    canvas_offsets=canvas_offsets)
print("Canvas-relative offsets")
display(canvas_relative_offsets)

In [ ]:
unblended_canvas=cr.render_unblended_project(
    project=mis_project,
    canvas_relative_offsets=canvas_relative_offsets,
    canvas_extents=canvas_extents)
display(unblended_canvas)

In [ ]:
blended_canvas_dfe=cr.render_blended_project(
    project=mis_project,
    canvas_relative_offsets=canvas_relative_offsets,
    canvas_extents=canvas_extents,
    weight=cr.weight_dfe)
display(blended_canvas_dfe)

In [ ]:
from misalign.calibration.scale_bar import image_with_scale_bar,scale_bar_calibrate,save_calibrated_image
from PIL.Image import Transpose
%matplotlib widget

In [ ]:
selected_image=blended_canvas_dfe
selected_image=selected_image.transpose(Transpose.ROTATE_90)
selected_calibration=mis_project.get_calibration()

In [ ]:
image_with_scale_bar(
    image=selected_image,
    scale_measurement="1mm",
    calibration=selected_calibration,
    loc="upper left")
scaled_dpi=1000
scale_bar_calibrate(scale_dpi=scaled_dpi)

In [ ]:
scaled_dpi=1000
# Increasing this DPI makes the above image smaller
# As a result, increasing the DPI makes the scale bar text/box larger relative to the image.
# Changing this DPI does not effect the resolution of the final image
scale_bar_calibrate(scale_dpi=scaled_dpi)

In [ ]:
save_calibrated_image(
    image_filepath="project_a_hdf5-scale.png",
    scale_dpi=scaled_dpi)